# Trening — EfficientNet-B4 — ISIC 2019+2020 (mel vs nevus)

Notebook treningowy oparty na wnioskach z EDA:
- Usuwamy wykryte duplikaty (11 w mel, 37 w nevus)
- Class weighting z policzonymi wagami (w_malignant ~1.12, w_benign ~0.90)
- Augmentacja korygująca różnicę rozdzielczości/jakości między klasami (nevus miał ~3x większą rozdzielczość niż mel)
- Split train/val zrobiony PRZED augmentacją, żeby uniknąć leakage
- Checkpointing co epokę pod limit 9h/sesję na Kaggle
- Model: EfficientNet-B4 (timm), wejście 380x380, wagi wstępnie trenowane na ImageNet

**Jak wznowić trening po przerwaniu sesji (ważne!):**
Jeśli sesja Kaggle się skończy (limit 9h) w trakcie treningu, ostatni zapisany checkpoint
jest w `/kaggle/working/checkpoints/`. Uruchom komórki 1-6 (setup, dane, model),
a w komórce treningowej ustaw `RESUME = True` — trening ruszy od ostatniej zapisanej epoki.


In [1]:
import os
import time
import copy
import random
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, confusion_matrix, roc_auc_score

!pip install timm --quiet
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Uzywane urzadzenie:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Uzywane urzadzenie: cuda
GPU: Tesla T4


## 1. Ścieżki i konfiguracja

Wszystkie kluczowe parametry treningu w jednym miejscu — łatwo zmienić bez szukania w kodzie.


In [2]:
BASE_DIR = Path("/kaggle/input/datasets/qikangdeng/isic-2019-and-2020-melanoma-dataset/isic19_20")
MEL_DIR = BASE_DIR / "mel"
NEVUS_DIR = BASE_DIR / "nevus"

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 380
BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
VAL_SPLIT = 0.2
RESUME = False
EARLY_STOP_PATIENCE = 5   # zatrzymaj jesli recall nie poprawia sie przez 5 epok

for p in [BASE_DIR, MEL_DIR, NEVUS_DIR]:
    print(p, "->", "OK" if p.exists() else "BRAK -- sprawdz sciezke")

/kaggle/input/datasets/qikangdeng/isic-2019-and-2020-melanoma-dataset/isic19_20 -> OK
/kaggle/input/datasets/qikangdeng/isic-2019-and-2020-melanoma-dataset/isic19_20/mel -> OK
/kaggle/input/datasets/qikangdeng/isic-2019-and-2020-melanoma-dataset/isic19_20/nevus -> OK


## 2. Wykrycie i usunięcie duplikatów

Z EDA wiemy, że są duplikaty wewnątrz klas (11 w mel, 37 w nevus). Liczymy je tu jeszcze raz
(żeby notebook był samodzielny, niezależny od EDA), i z każdej grupy duplikatów zostawiamy
tylko jeden plik.


In [3]:
def file_md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()

def dedupe(files):
    hashes = {}
    for f in files:
        h = file_md5(f)
        hashes.setdefault(h, []).append(f)
    kept = [group[0] for group in hashes.values()]
    n_removed = len(files) - len(kept)
    return kept, n_removed

mel_files_raw = list(MEL_DIR.glob("*.jpg"))
nevus_files_raw = list(NEVUS_DIR.glob("*.jpg"))

mel_files, mel_removed = dedupe(mel_files_raw)
nevus_files, nevus_removed = dedupe(nevus_files_raw)

print(f"mel: {len(mel_files_raw)} -> {len(mel_files)} (usunieto {mel_removed} duplikatow)")
print(f"nevus: {len(nevus_files_raw)} -> {len(nevus_files)} (usunieto {nevus_removed} duplikatow)")


mel: 5106 -> 5095 (usunieto 11 duplikatow)
nevus: 6343 -> 6306 (usunieto 37 duplikatow)


## 3. Budowa listy danych i podział train/val

Łączymy obie klasy w jedną tabelę z etykietą (0 = benign/nevus, 1 = malignant/mel),
a potem dzielimy na train/val ZE STRATYFIKACJĄ (czyli proporcja klas jest taka sama
w obu podzbiorach) — to ważne przy class imbalance.


In [4]:
records = (
    [{"path": str(f), "label": 1} for f in mel_files] +      # 1 = malignant
    [{"path": str(f), "label": 0} for f in nevus_files]       # 0 = benign
)
df = pd.DataFrame(records)
print("Razem obrazow po deduplikacji:", len(df))
print(df['label'].value_counts())

train_df, val_df = train_test_split(
    df, test_size=VAL_SPLIT, stratify=df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)} ({train_df['label'].mean()*100:.1f}% malignant)")
print(f"Val:   {len(val_df)} ({val_df['label'].mean()*100:.1f}% malignant)")


Razem obrazow po deduplikacji: 11401
label
0    6306
1    5095
Name: count, dtype: int64

Train: 9120 (44.7% malignant)
Val:   2281 (44.7% malignant)


## 4. Wagi klas (class weighting)

Liczymy dokładnie tak jak w EDA, ale teraz na podstawie zbioru TRENINGOWEGO
(po deduplikacji i splicie) — to najbardziej aktualne liczby.


In [5]:
n_benign_train = (train_df['label'] == 0).sum()
n_malignant_train = (train_df['label'] == 1).sum()
total_train = len(train_df)

w_benign = total_train / (2 * n_benign_train)
w_malignant = total_train / (2 * n_malignant_train)

print(f"w_benign = {w_benign:.4f}")
print(f"w_malignant = {w_malignant:.4f}")

# pos_weight dla BCEWithLogitsLoss to stosunek wagi klasy pozytywnej do negatywnej
pos_weight = torch.tensor([w_malignant / w_benign], dtype=torch.float32).to(device)
print(f"pos_weight (do BCEWithLogitsLoss) = {pos_weight.item():.4f}")


w_benign = 0.9040
w_malignant = 1.1187
pos_weight (do BCEWithLogitsLoss) = 1.2375


## 5. Augmentacje (transformacje obrazów)

Tu adresujemy wniosek z EDA: klasa nevus miała ~3x większą rozdzielczość niż mel,
co grozi tym, że model uczy się rozróżniać klasy po jakości obrazu, nie po treści.

**Strategia:** dodajemy do obu klas losowe, umiarkowane pogorszenie jakości
(blur, kompresja przez zmianę rozdzielczości w dół i z powrotem w górę) — to
częściowo ujednolica "wygląd" jakości między klasami, zamiast tylko czyścić jedną z nich.

Dla treningu używamy też standardowych augmentacji (obrót, odbicie, zmiana jasności) —
zdjęcia skóry nie mają "góry/dołu", więc swobodnie możemy je obracać i odbijać.


In [6]:
class RandomQualityDegradation:
    """Losowo pogarsza jakosc obrazu (symuluje nizsza rozdzielczosc), zeby
    zmniejszyc sygnal 'jakosc obrazu = klasa' miedzy mel i nevus."""
    def __init__(self, p=0.3, scale_range=(0.3, 0.7)):
        self.p = p
        self.scale_range = scale_range

    def __call__(self, img):
        if random.random() < self.p:
            w, h = img.size
            scale = random.uniform(*self.scale_range)
            small = img.resize((max(1, int(w*scale)), max(1, int(h*scale))), Image.BILINEAR)
            img = small.resize((w, h), Image.BILINEAR)
        return img

train_transform = transforms.Compose([
    RandomQualityDegradation(p=0.3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # standard ImageNet
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Transformacje gotowe.")


Transformacje gotowe.


## 6. Dataset i DataLoader

`Dataset` to klasa PyTorcha mówiąca "jak wczytać jeden przykład". `DataLoader` bierze
wiele przykładów naraz (batch) i podaje je do modelu podczas treningu.


In [7]:
class MelanomaDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return img, label

train_dataset = MelanomaDataset(train_df, transform=train_transform)
val_dataset = MelanomaDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# szybki test - wczytaj jeden batch, sprawdz ksztalty
imgs, labels = next(iter(train_loader))
print("Ksztalt batcha obrazow:", imgs.shape)
print("Ksztalt batcha etykiet:", labels.shape)


Train batches: 570, Val batches: 143
Ksztalt batcha obrazow: torch.Size([16, 3, 380, 380])
Ksztalt batcha etykiet: torch.Size([16])


## 7. Model — EfficientNet-B4

Ładujemy EfficientNet-B4 z wagami wstępnie wytrenowanymi na ImageNet (transfer learning),
i podmieniamy ostatnią warstwę na pojedynczy neuron (klasyfikacja binarna: malignant / benign).


In [8]:
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=1)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: EfficientNet-B4, {n_params/1e6:.1f}M parametrow")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Model: EfficientNet-B4, 17.6M parametrow


## 8. Funkcje pomocnicze do checkpointingu

Kluczowe pod limit 9h sesji na Kaggle — zapisujemy stan po każdej epoce, żeby móc wznowić
trening w nowej sesji bez zaczynania od zera.


In [9]:
LAST_CKPT = CHECKPOINT_DIR / "last_checkpoint.pt"
BEST_CKPT = CHECKPOINT_DIR / "best_model.pt"

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_recall):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_recall': best_recall,
    }, path)

def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    return ckpt['epoch'], ckpt['best_recall']

start_epoch = 0
best_recall = 0.0

if RESUME and LAST_CKPT.exists():
    start_epoch, best_recall = load_checkpoint(LAST_CKPT, model, optimizer, scheduler)
    start_epoch += 1
    print(f"Wznowiono trening od epoki {start_epoch}, najlepszy recall dotychczas: {best_recall:.4f}")
else:
    print("Zaczynamy trening od poczatku (epoka 0).")


Zaczynamy trening od poczatku (epoka 0).


## 9. Funkcje treningu i walidacji

`train_one_epoch` przechodzi raz przez cały zbiór treningowy i aktualizuje wagi modelu.
`validate` sprawdza jakość modelu na zbiorze walidacyjnym (bez aktualizacji wag) —
liczymy recall, precision i AUC, ze szczególnym naciskiem na recall dla malignant
(to nasz główny cel — mniej przeoczonych przypadków złośliwych).


In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels_dev = imgs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(imgs)
            loss = criterion(outputs, labels_dev)
            running_loss += loss.item() * imgs.size(0)

            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy().flatten())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    all_preds = (all_probs >= 0.5).astype(int)

    val_loss = running_loss / len(loader.dataset)
    recall = recall_score(all_labels, all_preds, pos_label=1)
    precision = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)

    return val_loss, recall, precision, auc, cm


## 10. Główna pętla treningowa

To jest właściwy trening — przechodzimy przez `NUM_EPOCHS` epok, po każdej:
1. Trenujemy na train_loader
2. Walidujemy na val_loader
3. Zapisujemy checkpoint (zawsze "ostatni", i osobno "najlepszy" jeśli recall się poprawił)
4. Dostrajamy learning rate, jeśli recall przestaje rosnąć (scheduler)

**Uwaga:** to jest komórka, która realnie długo się wykonuje (może kilka godzin
zależnie od liczby epok). Kaggle pokaże pasek postępu przez print po każdej epoce.


In [11]:
print(f"Start treningu: epoki {start_epoch} do {NUM_EPOCHS-1}")
session_start = time.time()
MAX_SESSION_SECONDS = 8.5 * 3600
epochs_without_improvement = 0

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, recall, precision, auc, cm = validate(model, val_loader, criterion, device)
    scheduler.step(recall)

    epoch_time = time.time() - epoch_start
    print(f"Epoka {epoch+1}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
          f"recall={recall:.4f} | precision={precision:.4f} | auc={auc:.4f} | "
          f"czas={epoch_time:.0f}s")
    print(f"  Confusion matrix:\n{cm}")

    save_checkpoint(LAST_CKPT, model, optimizer, scheduler, epoch, best_recall)

    if recall > best_recall:
        best_recall = recall
        epochs_without_improvement = 0
        save_checkpoint(BEST_CKPT, model, optimizer, scheduler, epoch, best_recall)
        print(f"  -> Nowy najlepszy recall: {best_recall:.4f}, zapisano best_model.pt")
    else:
        epochs_without_improvement += 1
        print(f"  -> Brak poprawy przez {epochs_without_improvement} epok(e/i)")

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping: brak poprawy recall przez {EARLY_STOP_PATIENCE} epok. "
              f"Zatrzymuje trening na epoce {epoch}. Najlepszy recall: {best_recall:.4f}")
        break

    elapsed = time.time() - session_start
    if elapsed > MAX_SESSION_SECONDS:
        print(f"\nZblizamy sie do limitu czasu sesji ({elapsed/3600:.1f}h). "
              f"Zatrzymuje trening na epoce {epoch}. Uruchom notebook ponownie z RESUME=True.")
        break

print("\nTrening zakonczony (lub przerwany bezpiecznie). Najlepszy recall:", best_recall)

Start treningu: epoki 0 do 19
Epoka 1/20 | train_loss=0.3599 | val_loss=0.2597 | recall=0.8862 | precision=0.9158 | auc=0.9645 | czas=778s
  Confusion matrix:
[[1179   83]
 [ 116  903]]
  -> Nowy najlepszy recall: 0.8862, zapisano best_model.pt
Epoka 2/20 | train_loss=0.2534 | val_loss=0.2433 | recall=0.8705 | precision=0.9446 | auc=0.9702 | czas=783s
  Confusion matrix:
[[1210   52]
 [ 132  887]]
  -> Brak poprawy przez 1 epok(e/i)
Epoka 3/20 | train_loss=0.2127 | val_loss=0.2119 | recall=0.9117 | precision=0.9235 | auc=0.9759 | czas=789s
  Confusion matrix:
[[1185   77]
 [  90  929]]
  -> Nowy najlepszy recall: 0.9117, zapisano best_model.pt
Epoka 4/20 | train_loss=0.1823 | val_loss=0.2074 | recall=0.8950 | precision=0.9550 | auc=0.9771 | czas=803s
  Confusion matrix:
[[1219   43]
 [ 107  912]]
  -> Brak poprawy przez 1 epok(e/i)
Epoka 5/20 | train_loss=0.1546 | val_loss=0.2160 | recall=0.9166 | precision=0.9148 | auc=0.9743 | czas=795s
  Confusion matrix:
[[1175   87]
 [  85  934]]


## 11. Podsumowanie i zapis finalnego modelu

Po zakończeniu treningu (lub gdy jesteś zadowolony z wyników po kilku epokach),
zapisz finalny model w czytelnej formie do dalszego użycia (np. wgrania na Hugging Face,
tak jak Twoje wcześniejsze modele).


In [12]:
# Wczytaj najlepszy model (wg recall) do finalnej oceny / eksportu
best_epoch, best_recall_loaded = load_checkpoint(BEST_CKPT, model, optimizer, scheduler)
print(f"Wczytano najlepszy model z epoki {best_epoch}, recall={best_recall_loaded:.4f}")

# Finalna walidacja z najlepszym modelem
val_loss, recall, precision, auc, cm = validate(model, val_loader, criterion, device)
print(f"\nFinalne metryki (best model):")
print(f"  Recall (malignant):    {recall:.4f}")
print(f"  Precision (malignant): {precision:.4f}")
print(f"  AUC:                   {auc:.4f}")
print(f"  Confusion matrix:\n{cm}")

# Zapis samego state_dict modelu (mniejszy plik, gotowy do dalszego uzycia/eksportu)
FINAL_MODEL_PATH = Path("/kaggle/working/efficientnet_b4_melanoma_final.pt")
torch.save(model.state_dict(), FINAL_MODEL_PATH)
print(f"\nZapisano finalny model: {FINAL_MODEL_PATH}")


Wczytano najlepszy model z epoki 7, recall=0.9342

Finalne metryki (best model):
  Recall (malignant):    0.9342
  Precision (malignant): 0.8964
  AUC:                   0.9798
  Confusion matrix:
[[1152  110]
 [  67  952]]

Zapisano finalny model: /kaggle/working/efficientnet_b4_melanoma_final.pt


In [13]:
import os

print("Zawartosc /kaggle/working:")
for item in os.listdir('/kaggle/working'):
    print(" -", item)

print("\nZawartosc /kaggle/working/checkpoints:")
if os.path.exists('/kaggle/working/checkpoints'):
    for item in os.listdir('/kaggle/working/checkpoints'):
        path = os.path.join('/kaggle/working/checkpoints', item)
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f" - {item} ({size_mb:.1f} MB)")
else:
    print("Folder checkpoints nie istnieje!")

final_path = '/kaggle/working/efficientnet_b4_melanoma_final.pt'
if os.path.exists(final_path):
    size_mb = os.path.getsize(final_path) / (1024*1024)
    print(f"\nFinalny model: OK ({size_mb:.1f} MB)")
else:
    print("\nFinalny model: BRAK")

Zawartosc /kaggle/working:
 - __notebook__.ipynb
 - checkpoints
 - efficientnet_b4_melanoma_final.pt

Zawartosc /kaggle/working/checkpoints:
 - best_model.pt (201.9 MB)
 - last_checkpoint.pt (201.9 MB)

Finalny model: OK (67.7 MB)


## Notatki

- Jeśli w trakcie treningu wyskoczy `CUDA out of memory` — zmniejsz `BATCH_SIZE` (np. z 16 na 8) w komórce konfiguracji i uruchom ponownie od tamtego miejsca.
- Jeśli sesja się przerwie (limit 9h) — otwórz notebook ponownie, ustaw `RESUME = True` w komórce konfiguracji, uruchom wszystkie komórki po kolei (setup → dane → model → checkpointing), trening ruszy od zapisanej epoki.
- `best_model.pt` zawsze trzyma wagi z epoki o najwyższym recall na malignant — to model do dalszego użycia, nie ostatnia epoka.
- Po zakończeniu warto porównać ten model z obecnym EfficientNet-B0 na tych samych danych walidacyjnych, żeby mieć uczciwe porównanie recall/precision.
